### Building a passing network

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

In [3]:
df = pd.read_parquet('../data/all_events_2015_16.parquet')

### As I'm looking at passes, I need to filter down the data only to pass events

In [4]:
Pass = df[df['type'] == 'Pass']

### Need to filter down to completed passes to a known recipient

In [5]:
Pass = Pass[Pass['pass_outcome'].isna() & Pass['pass_recipient'].notna()]


In [6]:
matchone = Pass[Pass['match_id'] == 3754217]

### Edges can be modelled as passes, and nodes as the players, weight of edge = No. passes

In [7]:
edges = matchone.groupby(['player', 'pass_recipient']).size().reset_index(name='count')
edges.head()

,player,pass_recipient,count
0,Aaron Ramsey,Alex Oxlade-Chamberlain,1
1,Aaron Ramsey,Alexis Alejandro Sánchez Sánchez,1
2,Aaron Ramsey,Calum Chambers,2
3,Aaron Ramsey,Francis Joseph Coquelin,1
4,Aaron Ramsey,Gabriel Armando de Abreu,2


### Learning the basics for Programming a network

In [8]:
# Building a directed graph
graph = nx.from_pandas_edgelist(edges, 'player', 'pass_recipient', ['count'], create_using=nx.DiGraph())

In [9]:
graph.nodes()
graph.edges(data=True)

OutEdgeDataView([('Aaron Ramsey', 'Alex Oxlade-Chamberlain', {'count': 1}), ('Aaron Ramsey', 'Alexis Alejandro Sánchez Sánchez', {'count': 1}), ('Aaron Ramsey', 'Calum Chambers', {'count': 2}), ('Aaron Ramsey', 'Francis Joseph Coquelin', {'count': 1}), ('Aaron Ramsey', 'Gabriel Armando de Abreu', {'count': 2}), ('Aaron Ramsey', 'Héctor Bellerín Moruno', {'count': 3}), ('Aaron Ramsey', 'Ignacio Monreal Eraso', {'count': 5}), ('Aaron Ramsey', 'Laurent Koscielny', {'count': 3}), ('Aaron Ramsey', 'Mesut Özil', {'count': 5}), ('Aaron Ramsey', 'Petr Čech', {'count': 1}), ('Aaron Ramsey', 'Santiago Cazorla González', {'count': 4}), ('Aaron Ramsey', 'Theo Walcott', {'count': 3}), ('Alex Oxlade-Chamberlain', 'Héctor Bellerín Moruno', {'count': 2}), ('Alex Oxlade-Chamberlain', 'Olivier Giroud', {'count': 1}), ('Alex Oxlade-Chamberlain', 'Santiago Cazorla González', {'count': 1}), ('Alexis Alejandro Sánchez Sánchez', 'Aaron Ramsey', {'count': 5}), ('Alexis Alejandro Sánchez Sánchez', 'Francis Jos

#### We're dealing with a directed graph, and its important to note the indegree and outdegree of a player to see if they distribute alot and/or recieve alot

In [10]:
In = nx.in_degree_centrality(graph)
Out = nx.out_degree_centrality(graph)

In [11]:
import numpy as np
print(np.c_[Out, In])


[[{'Aaron Ramsey': 0.4444444444444444, 'Alex Oxlade-Chamberlain': 0.1111111111111111, 'Alexis Alejandro Sánchez Sánchez': 0.2962962962962963, 'Calum Chambers': 0.18518518518518517, 'Francis Joseph Coquelin': 0.25925925925925924, 'Gabriel Armando de Abreu': 0.25925925925925924, 'Héctor Bellerín Moruno': 0.37037037037037035, 'Ignacio Monreal Eraso': 0.25925925925925924, 'Laurent Koscielny': 0.2962962962962963, 'Mesut Özil': 0.2962962962962963, 'Petr Čech': 0.3333333333333333, 'Santiago Cazorla González': 0.37037037037037035, 'Theo Walcott': 0.2222222222222222, 'Olivier Giroud': 0.14814814814814814, 'Asmir Begović': 0.25925925925925924, 'Branislav Ivanović': 0.3333333333333333, 'César Azpilicueta Tanco': 0.4444444444444444, 'Diego da Silva Costa': 0.25925925925925924, 'Francesc Fàbregas i Soler': 0.4074074074074074, 'Gary Cahill': 0.25925925925925924, 'Kurt Happy Zouma': 0.37037037037037035, 'Oscar dos Santos Emboaba Júnior': 0.25925925925925924, 'Eden Hazard': 0.37037037037037035, 'John 

### Passes for the individual teams in a match, which I'll later generalise for all matches

In [12]:
team_graphs = {}

for team_name, team_df in matchone.groupby('team'):
    edges = team_df.groupby(['player', 'pass_recipient']).size().reset_index(name='count')
    team_graphs[team_name] = nx.from_pandas_edgelist(edges, 'player', 'pass_recipient', ['count'], create_using=nx.DiGraph())
    

In [13]:
team_degrees = {}

for team_name, team_graph in team_graphs.items():
    deg = nx.degree_centrality(team_graph)
    in_deg = nx.in_degree_centrality(team_graph)
    out_deg = nx.out_degree_centrality(team_graph)
    team_degrees[team_name] = (deg, in_deg, out_deg)

In [14]:
# Arsenal's in degree centrality
team_degrees['Arsenal'][1]

{'Aaron Ramsey': 0.9230769230769231,
 'Alex Oxlade-Chamberlain': 0.38461538461538464,
 'Alexis Alejandro Sánchez Sánchez': 0.6923076923076923,
 'Calum Chambers': 0.3076923076923077,
 'Francis Joseph Coquelin': 0.5384615384615385,
 'Gabriel Armando de Abreu': 0.38461538461538464,
 'Héctor Bellerín Moruno': 0.7692307692307693,
 'Ignacio Monreal Eraso': 0.7692307692307693,
 'Laurent Koscielny': 0.6923076923076923,
 'Mesut Özil': 0.6923076923076923,
 'Petr Čech': 0.23076923076923078,
 'Santiago Cazorla González': 0.8461538461538463,
 'Theo Walcott': 0.6153846153846154,
 'Olivier Giroud': 0.15384615384615385}

### Putting each teams  into a dataframe now for inspection, make sure everything looks good

In [15]:
def network(team_name):
    player = team_degrees[team_name][0]
    in_deg = team_degrees[team_name][1]
    out_deg = team_degrees[team_name][2]
    return pd.DataFrame({
        'player': player.keys(),
        'degree_centrality': player.values(),
        'in_degree_centrality': in_deg.values(),
        'out_degree_centrality': out_deg.values()
    })


In [16]:
network('Arsenal')

,player,degree_centrality,in_degree_centrality,out_degree_centrality
0,Aaron Ramsey,1.846154,0.923077,0.923077
1,Alex Oxlade-Chamberlain,0.615385,0.384615,0.230769
2,Alexis Alejandro Sánchez Sánchez,1.307692,0.692308,0.615385
3,Calum Chambers,0.692308,0.307692,0.384615
4,Francis Joseph Coquelin,1.076923,0.538462,0.538462
5,Gabriel Armando de Abreu,0.923077,0.384615,0.538462
6,Héctor Bellerín Moruno,1.538462,0.769231,0.769231
7,Ignacio Monreal Eraso,1.307692,0.769231,0.538462
8,Laurent Koscielny,1.307692,0.692308,0.615385
9,Mesut Özil,1.307692,0.692308,0.615385


### Looks good! Now we can be sure the output is what we want it to look like. From here, we can do this for every team in every match of the season, in one big loop

In [47]:
all_match_degrees = {} # Dictionary to store team degrees for each match

for match_id in Pass['match_id'].unique():
    match_passes = Pass[Pass['match_id'] == match_id]
    team_graphs = {}
    for team_name, team_df in match_passes.groupby('team'):
        edges = team_df.groupby(['player', 'pass_recipient']).size().reset_index(name='count')
        team_graphs[team_name] = nx.from_pandas_edgelist(edges, 'player', 'pass_recipient', ['count'], create_using=nx.DiGraph())
    team_degrees = {}
    for team_name, team_graph in team_graphs.items():
        deg = nx.degree_centrality(team_graph)
        in_deg = nx.in_degree_centrality(team_graph)
        out_deg = nx.out_degree_centrality(team_graph)
        team_degrees[team_name] = (deg, in_deg, out_deg)
    all_match_degrees[match_id] = team_degrees # Team degrees for each match stored in a dictionary with match_id as key

len(all_match_degrees)


380

## Feature Matrix Construction
#### To make this ready for further analysis and ML, we must find an approach to collapse `all_match_degrees` into a flat 760-row DataFrame (a row each for 2 teams in a 380 match season) with summary degree centrality statistics as columns. 

#### From there we can compute centralisation index* per team per match and join both into a single feature matrix.

\* The centralisation index measures how concentrated passing is around a single player, where 
a high value indicates one player dominates distribution, a low value indicates passing 
is spread evenly across the team.

In [18]:
rows = []
for match_id, degrees in all_match_degrees.items():
    for team_name, (deg, in_deg, out_deg) in degrees.items():
        for player in deg.keys():
            rows.append({
                'match_id': match_id,
                'team': team_name,
                'player': player,
                'degree_centrality': deg[player],
                'in_degree_centrality': in_deg[player],
                'out_degree_centrality': out_deg[player]
            })
player_centrality = pd.DataFrame(rows)

In [19]:
player_centrality.head()

,match_id,team,player,degree_centrality,in_degree_centrality,out_degree_centrality
0,3754217,Arsenal,Aaron Ramsey,1.846154,0.923077,0.923077
1,3754217,Arsenal,Alex Oxlade-Chamberlain,0.615385,0.384615,0.230769
2,3754217,Arsenal,Alexis Alejandro Sánchez Sánchez,1.307692,0.692308,0.615385
3,3754217,Arsenal,Calum Chambers,0.692308,0.307692,0.384615
4,3754217,Arsenal,Francis Joseph Coquelin,1.076923,0.538462,0.538462


### Condensing down to team data, resulting in degree averages for both teams from each match in the season => 760 rows

In [20]:
team_centrality = player_centrality.groupby(['match_id', 'team']).agg({
    'degree_centrality': 'mean',
    'in_degree_centrality': 'mean',
    'out_degree_centrality': 'mean'
}).reset_index()
team_centrality.shape

(760, 5)

In [21]:
team_centrality.head()

,match_id,team,degree_centrality,in_degree_centrality,out_degree_centrality
0,3753972,AFC Bournemouth,1.087912,0.543956,0.543956
1,3753972,Swansea City,1.252747,0.626374,0.626374
2,3753973,Chelsea,1.406593,0.703297,0.703297
3,3753973,West Ham United,1.186813,0.593407,0.593407
4,3753974,Everton,1.230769,0.615385,0.615385


#### The centralisation index is given as:
$$C = \frac{\sum_{i=1}^{n} (p_{max} - p_i)}{p_{max} \cdot (n-1) \cdot n}$$

Where
- $p_i$ = number of completed passes by player $i$
- $p_{max}$ = maximum passes by any single player in the team
- $n$ = number of players in the team

In [22]:
# Calculating the centralisation index per team per match
# Credit to Soccermatics for the code used

def centralisation_index(Pass):
    no_passes = Pass.groupby(['player']).pass_recipient.count().reset_index()
    no_passes.rename({'pass_recipient':'pass_count'}, axis='columns', inplace=True)
    #find one who made most passes
    max_no = no_passes["pass_count"].max()
    #calculate the denominator - 10*the total sum of passes
    denominator = 10*no_passes["pass_count"].sum()
    nominator = (max_no - no_passes["pass_count"]).sum()
    centralisation_index = nominator/denominator

    return centralisation_index

In [23]:
ciperteam = Pass.groupby(['match_id', 'team']).apply(centralisation_index).reset_index(name='centralisation_index')
ciperteam.head()

,match_id,team,centralisation_index
0,3753972,AFC Bournemouth,0.076509
1,3753972,Swansea City,0.095152
2,3753973,Chelsea,0.094360
3,3753973,West Ham United,0.086667
4,3753974,Everton,0.103689


In [24]:
Team_centrality = pd.merge(team_centrality, ciperteam, on=['match_id', 'team'])
Team_centrality.head()

,match_id,team,degree_centrality,in_degree_centrality,out_degree_centrality,centralisation_index
0,3753972,AFC Bournemouth,1.087912,0.543956,0.543956,0.076509
1,3753972,Swansea City,1.252747,0.626374,0.626374,0.095152
2,3753973,Chelsea,1.406593,0.703297,0.703297,0.094360
3,3753973,West Ham United,1.186813,0.593407,0.593407,0.086667
4,3753974,Everton,1.230769,0.615385,0.615385,0.103689


### Time for the xG data, very important for the final model. We'll use StatsBomb's evaluation of the total xG for a team in a match, they know what they're doing

In [25]:
xg = df[df['type'] == 'Shot'][['match_id', 'team', 'shot_statsbomb_xg']].groupby(['match_id', 'team']).sum().reset_index()
xg.head()

,match_id,team,shot_statsbomb_xg
0,3753972,AFC Bournemouth,1.790883
1,3753972,Swansea City,2.146323
2,3753973,Chelsea,2.405319
3,3753973,West Ham United,0.886418
4,3753974,Everton,1.806268


In [26]:
# Running out of names here, so I'll just call it "Team_centrality_xg"
Team_centrality_xg = pd.merge(Team_centrality, xg, on=['match_id', 'team'])
Team_centrality_xg.head()

,match_id,team,degree_centrality,in_degree_centrality,out_degree_centrality,centralisation_index,shot_statsbomb_xg
0,3753972,AFC Bournemouth,1.087912,0.543956,0.543956,0.076509,1.790883
1,3753972,Swansea City,1.252747,0.626374,0.626374,0.095152,2.146323
2,3753973,Chelsea,1.406593,0.703297,0.703297,0.094360,2.405319
3,3753973,West Ham United,1.186813,0.593407,0.593407,0.086667,0.886418
4,3753974,Everton,1.230769,0.615385,0.615385,0.103689,1.806268


### Defensive metrics too, I can make this one on my own..

In [27]:
df['type'].unique()

<ArrowStringArray>
[      'Starting XI',        'Half Start',              'Pass',
     'Ball Receipt*',             'Carry',          'Pressure',
             'Block',     'Ball Recovery',     'Dribbled Past',
           'Dribble',    'Foul Committed',          'Foul Won',
        'Miscontrol',              'Duel',              'Shot',
       'Goal Keeper',         'Clearance',            'Shield',
      'Interception',      'Dispossessed',   'Injury Stoppage',
        'Player Off',         'Player On',     'Bad Behaviour',
 'Referee Ball-Drop',    'Tactical Shift',          'Half End',
      'Substitution',             'Error',      'Own Goal For',
  'Own Goal Against',             '50/50',           'Offside']
Length: 33, dtype: str

In [28]:
# Focusing on defensive events ['Pressure', 'Interception', 'Block', 'Clearance']
defensive_events = ['Pressure', 'Interception', 'Block', 'Clearance']
defensive_counts = df[df['type'].isin(defensive_events)].groupby(['match_id', 'team', 'type']).size().unstack(fill_value=0).reset_index()
defensive_counts.head()

type,match_id,team,Block,Clearance,Interception,Pressure
0,3753972,AFC Bournemouth,20,35,10,132
1,3753972,Swansea City,31,31,3,236
2,3753973,Chelsea,15,21,8,154
3,3753973,West Ham United,24,41,25,154
4,3753974,Everton,18,8,8,165


In [29]:
Team_Centrality = pd.merge(Team_centrality_xg, defensive_counts, on=['match_id', 'team'])
Team_Centrality.columns

Index(['match_id', 'team', 'degree_centrality', 'in_degree_centrality',
       'out_degree_centrality', 'centralisation_index', 'shot_statsbomb_xg',
       'Block', 'Clearance', 'Interception', 'Pressure'],
      dtype='str')

In [55]:
Team_Centrality.head()

,match_id,team,degree_centrality,in_degree_centrality,out_degree_centrality,centralisation_index,shot_statsbomb_xg,Block,Clearance,Interception,Pressure
0,3753972,AFC Bournemouth,1.087912,0.543956,0.543956,0.076509,1.790883,20,35,10,132
1,3753972,Swansea City,1.252747,0.626374,0.626374,0.095152,2.146323,31,31,3,236
2,3753973,Chelsea,1.406593,0.703297,0.703297,0.094360,2.405319,15,21,8,154
3,3753973,West Ham United,1.186813,0.593407,0.593407,0.086667,0.886418,24,41,25,154
4,3753974,Everton,1.230769,0.615385,0.615385,0.103689,1.806268,18,8,8,165


### Must now generate home and away teams as column features rather than 2 teams per 1 match in the rows.

In [48]:
# Same as in data_exploration.ipynb
from statsbombpy import sb
matches = sb.matches(competition_id=2, season_id=27)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [54]:
matches[['home_team', 'away_team', 'home_score', 'away_score', 'match_id']].head()

,home_team,away_team,home_score,away_score,match_id
0,Chelsea,Arsenal,2,0,3754217
1,Aston Villa,Arsenal,0,2,3754117
2,Arsenal,Manchester City,2,1,3754296
3,Swansea City,Arsenal,0,3,3753983
4,Arsenal,Sunderland,3,1,3754160


In [56]:
season = pd.merge(Team_Centrality, matches, on = ['match_id']).reset_index()

In [65]:
season['home'] = season['team'] == season['home_team']
season[['home', 'team']]
# True = team is home, False = team is away

,home,team
0,False,AFC Bournemouth
1,True,Swansea City
2,True,Chelsea
3,False,West Ham United
4,False,Everton
...,...,...
755,True,West Ham United
756,True,Everton
757,False,West Bromwich Albion
758,False,Crystal Palace


In [66]:
season = season[['match_id', 'team', 'home', 'home_score', 'away_score',
                'degree_centrality', 'in_degree_centrality',
                'out_degree_centrality', 'centralisation_index', 'shot_statsbomb_xg',
                'Block', 'Clearance', 'Interception', 'Pressure']]

In [72]:
season.head()

,match_id,team,home,home_score,away_score,degree_centrality,in_degree_centrality,out_degree_centrality,centralisation_index,shot_statsbomb_xg,Block,Clearance,Interception,Pressure,outcome
0,3753972,AFC Bournemouth,False,2,2,1.087912,0.543956,0.543956,0.076509,1.790883,20,35,10,132,match_id
1,3753972,Swansea City,True,2,2,1.252747,0.626374,0.626374,0.095152,2.146323,31,31,3,236,match_id
2,3753973,Chelsea,True,2,2,1.406593,0.703297,0.703297,0.094360,2.405319,15,21,8,154,match_id
3,3753973,West Ham United,False,2,2,1.186813,0.593407,0.593407,0.086667,0.886418,24,41,25,154,match_id
4,3753974,Everton,False,0,1,1.230769,0.615385,0.615385,0.103689,1.806268,18,8,8,165,match_id


In [83]:
# Match outcome classifier using np.select()
import numpy as np

conditions = [
    (season['home'] == True) & (season['home_score'] > season['away_score']), # Home team win
    (season['home'] == False) & (season['home_score'] < season['away_score']), # Away team win
    (season['home'] == True) & (season['home_score'] == season['away_score']), # Home team draw
    (season['home'] == False) & (season['home_score'] == season['away_score']), # Away team draw
    (season['home'] == True) & (season['home_score'] < season['away_score']), # Home team loss
    (season['home'] == False) & (season['home_score'] > season['away_score']) # Away team loss

]
outcomes = ['Win', 'Win', 'Draw', 'Draw', 'Loss', 'Loss']
season['outcome'] = np.select(conditions, outcomes, default='Unknown')


In [ ]:
# Checking for no 'Unknown' outcomes
season['outcome'].value_counts()

outcome
Win     273
Loss    273
Draw    214
Name: count, dtype: int64

In [85]:
season.columns.tolist()

['match_id',
 'team',
 'home',
 'home_score',
 'away_score',
 'degree_centrality',
 'in_degree_centrality',
 'out_degree_centrality',
 'centralisation_index',
 'shot_statsbomb_xg',
 'Block',
 'Clearance',
 'Interception',
 'Pressure',
 'outcome']

## We have our feature matrix ready! We have:
- #### Network metrics
- #### Centralisation index
- #### xG
- #### Defensive contributions (We can later figure out which ones are most meaningful)
- #### Match outcome (target variable for ML model)

In [86]:
season.to_parquet('../data/feature_matrix.parquet')